# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# 0.1 Mount Google Drive để lưu checkpoint/adapter đề phòng mất session
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 0.2 Đồng bộ mã nguồn repo nếu chạy trên Colab
import os
REPO_DIR = "/content/K4-Track-4-Day10-VisionLanguageModel"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/awun0105/K4-Track-4-Day10-VisionLanguageModel.git /content/K4-Track-4-Day10-VisionLanguageModel
%cd /content/K4-Track-4-Day10-VisionLanguageModel

In [ ]:
# 0.3 Kiểm tra T4 GPU và cài đặt thư viện
# Lưu ý: Chọn menu Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# 0.4 Cấu hình thư mục lưu checkpoint trên Google Drive
import shutil
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/AI20K_Lab_Day10_Checkpoints"
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

def backup_to_drive(src_path, exp_name):
    """Hàm tiện ích sao lưu checkpoint/adapter sang Google Drive"""
    dest = os.path.join(DRIVE_BACKUP_DIR, exp_name)
    if os.path.isdir(src_path):
        shutil.copytree(src_path, dest, dirs_exist_ok=True)
    elif os.path.isfile(src_path):
        shutil.copy2(src_path, dest)
    print(f"✅ Đã sao lưu {src_path} -> {dest}")

In [ ]:
# 0.5 Kaggle & Hugging Face credentials: nạp từ Colab Secrets hoặc file .env / biến môi trường
# TUÂN THỦ QUY ĐỊNH: Tuyệt đối KHÔNG ghi cứng token / secret key vào code.
import os

# 1. Nạp từ Colab Secrets (khi chạy trên Google Colab)
try:
    from google.colab import userdata
    for key in ["KAGGLE_USERNAME", "KAGGLE_KEY", "HF_TOKEN"]:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
        except Exception:
            pass
except ImportError:
    pass

# 2. Nạp từ file .env (khi chạy local / Google Drive mount có .env)
env_file = ".env"
if os.path.exists(env_file):
    try:
        with open(env_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    k, v = line.split("=", 1)
                    k, v = k.strip(), v.strip().split("#")[0].strip()
                    if k and v and k not in os.environ:
                        os.environ[k] = v
    except Exception:
        pass

# Đồng bộ token cho Hugging Face Hub
if os.environ.get("HF_TOKEN"):
    os.environ["HUGGINGFACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
    try:
        from huggingface_hub import login
        login(token=os.environ["HF_TOKEN"])
        print("✅ Đã xác thực Hugging Face Hub thành công.")
    except Exception as e:
        print(f"Lưu ý Hugging Face login: {e}")

print("Kaggle Username:", "Đã thiết lập" if os.environ.get("KAGGLE_USERNAME") else "Chưa có (cần đặt trong Colab Secrets)")
print("Kaggle Key:", "Đã thiết lập" if os.environ.get("KAGGLE_KEY") else "Chưa có (cần đặt trong Colab Secrets)")
print("HF Token:", "Đã thiết lập" if os.environ.get("HF_TOKEN") else "Chưa có (tuỳ chọn)")


## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
# YOUR CODE HERE: đọc data/splits/train.jsonl, xem vài mẫu, vẽ box lên ảnh
import json
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

with open("data/splits/train.jsonl", "r", encoding="utf-8") as f:
    samples = [json.loads(line) for line in f][:3]

for item in samples:
    img = Image.open(f"data/{item['image_path']}")
    draw = ImageDraw.Draw(img)
    w, h = item["width"], item["height"]
    for obj in item["objects"]:
        xc, yc, bw, bh = obj["yolo"]
        x1, y1 = (xc - bw/2) * w, (yc - bh/2) * h
        x2, y2 = (xc + bw/2) * w, (yc + bh/2) * h
        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
    plt.figure(figsize=(6, 4))
    plt.imshow(img)
    plt.title(f"Image ID: {item['image_id']} | Objects: {len(item['objects'])}")
    plt.axis("off")
    plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
import json
import math


def _scale_yolo_box(yolo: list[float], width: int, height: int) -> list[int]:
    if len(yolo) != 4:
        raise ValueError(f"YOLO box phải có 4 giá trị, nhận được {len(yolo)}")
    xc, yc, bw, bh = (float(value) for value in yolo)
    if not all(math.isfinite(value) for value in (xc, yc, bw, bh)):
        raise ValueError(f"YOLO box có giá trị không hữu hạn: {yolo}")
    if bw <= 0 or bh <= 0:
        raise ValueError(f"YOLO box phải có width/height dương: {yolo}")

    coords = [
        round((xc - bw / 2.0) * width),
        round((yc - bh / 2.0) * height),
        round((xc + bw / 2.0) * width),
        round((yc + bh / 2.0) * height),
    ]
    x1, y1, x2, y2 = [max(0, min(limit, value)) for value, limit in zip(coords, (width, height, width, height))]
    if x2 <= x1 or y2 <= y1:
        raise ValueError(f"Box suy biến sau khi kẹp vào ảnh: {yolo} -> {[x1, y1, x2, y2]}")
    return [x1, y1, x2, y2]


def yolo_to_box(label: str, yolo: list[float]) -> dict:
    """Chuyển YOLO [xc, yc, w, h] sang toạ độ grounding nguyên trên thang 0–1000."""
    return {"label": label, "bbox_2d": _scale_yolo_box(yolo, 1000, 1000)}


def yolo_to_pixel_box(label: str, yolo: list[float], width: int = 640, height: int = 360) -> dict:
    """Chuyển YOLO sang pixel [x1, y1, x2, y2] của ảnh có kích thước width × height."""
    return {"label": label, "bbox_2d": _scale_yolo_box(yolo, width, height)}


def objects_to_answer(
    objects: list[dict],
    coord_scale: str = "norm1000",
    width: int = 640,
    height: int = 360,
) -> str:
    """Đổi nhãn thành JSON ổn định theo thứ tự trên–dưới, trái–phải."""
    if coord_scale == "norm1000":
        boxes = [yolo_to_box(obj["label"], obj["yolo"]) for obj in objects]
    elif coord_scale == "pixel":
        boxes = [yolo_to_pixel_box(obj["label"], obj["yolo"], width, height) for obj in objects]
    else:
        raise ValueError(f"coord_scale không hợp lệ: {coord_scale}")

    boxes.sort(key=lambda box: (box["bbox_2d"][1], box["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"), ensure_ascii=False)

In [ ]:
SYSTEM_PROMPT = (
    "You are a road inspection assistant. Detect all road damage objects in the image. "
    "Return only a JSON list. Each item must have 'label' and 'bbox_2d'. "
    "Allowed labels: 'pothole', 'crack', 'manhole'. "
    "Coordinates must be integers on the 0–1000 scale in [x1, y1, x2, y2] order."
)

PIXEL_SYSTEM_PROMPT = (
    "You are a road inspection assistant. Detect all road damage objects in the image. "
    "Return only a JSON list. Each item must have 'label' and 'bbox_2d'. "
    "Allowed labels: 'pothole', 'crack', 'manhole'. "
    "Coordinates must be integer pixel values in [x1, y1, x2, y2] order, "
    "with x in [0, 640] and y in [0, 360]."
)


def system_prompt_for_scale(coord_scale: str) -> str:
    if coord_scale == "norm1000":
        return SYSTEM_PROMPT
    if coord_scale == "pixel":
        return PIXEL_SYSTEM_PROMPT
    raise ValueError(f"coord_scale không hợp lệ: {coord_scale}")

## 3. E0: zero-shot

In [ ]:
# Nạp model 4-bit cho E0 và dùng lại cùng cấu hình khi huấn luyện/đánh giá E1–E3.
import json
import math
from pathlib import Path

import torch
from PIL import Image
from tqdm import tqdm
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration

model_id = "Qwen/Qwen3-VL-4B-Instruct"
processor = AutoProcessor.from_pretrained(model_id)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    # Theo GUIDE: giữ visual và lm_head không bị lượng tử hoá.
    llm_int8_skip_modules=["visual", "lm_head"],
)


def load_base_model():
    if not torch.cuda.is_available():
        raise RuntimeError("Lab yêu cầu GPU CUDA; hãy chọn runtime T4 trên Colab.")
    return Qwen3VLForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map={"": 0},
    )


def _chunks(items, batch_size: int):
    if batch_size < 1:
        raise ValueError("batch_size phải lớn hơn 0")
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


def write_predictions_jsonl(
    infer_model,
    entries: list[dict],
    output_path: str | Path,
    prompt_text: str,
    batch_size: int = 1,
    max_new_tokens: int = 256,
    desc: str = "Suy luận",
) -> Path:
    """Sinh JSONL theo batch; padding trái giúp lấy đúng phần token model mới sinh."""
    if batch_size < 1:
        raise ValueError("batch_size phải lớn hơn 0")
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    batches = math.ceil(len(entries) / batch_size)

    with output_path.open("w", encoding="utf-8") as out_file:
        for batch_entries in tqdm(_chunks(entries, batch_size), total=batches, desc=desc):
            images = []
            for item in batch_entries:
                with Image.open(Path("data") / item["image_path"]) as source_image:
                    images.append(source_image.convert("RGB"))

            messages = [
                [{"role": "user", "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt_text},
                ]}]
                for image in images
            ]
            inputs = processor.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt",
                padding=True,
            ).to("cuda")
            prompt_width = inputs["input_ids"].shape[1]

            with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16):
                generated = infer_model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                )
            raw_outputs = processor.batch_decode(
                generated[:, prompt_width:],
                skip_special_tokens=True,
            )
            for item, raw in zip(batch_entries, raw_outputs, strict=True):
                out_file.write(json.dumps(
                    {"image_id": item["image_id"], "raw": raw}, ensure_ascii=False
                ) + "\n")

    return output_path


model = load_base_model()
model.eval()

with open("data/splits/test.jsonl", "r", encoding="utf-8") as f:
    test_entries = [json.loads(line) for line in f]

write_predictions_jsonl(
    model,
    test_entries,
    "work/e0.jsonl",
    SYSTEM_PROMPT,
    batch_size=4,
    max_new_tokens=256,
    desc="Evaluating E0 Zero-shot",
)

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# Dataset chat-template có mask loss trên prompt; collator giữ đủ các tensor đa phương thức.
import json
from pathlib import Path

import torch
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
from PIL import Image


class RoadDamageDataset(Dataset):
    def __init__(self, jsonl_path, processor, coord_scale: str = "norm1000"):
        if coord_scale not in ("norm1000", "pixel"):
            raise ValueError(f"coord_scale không hợp lệ: {coord_scale}")
        with open(jsonl_path, "r", encoding="utf-8") as f:
            self.entries = [json.loads(line) for line in f]
        self.processor = processor
        self.coord_scale = coord_scale

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        item = self.entries[idx]
        with Image.open(Path("data") / item["image_path"]) as source_image:
            image = source_image.convert("RGB")

        answer = objects_to_answer(
            item["objects"],
            coord_scale=self.coord_scale,
            width=item["width"],
            height=item["height"],
        )
        user_message = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": system_prompt_for_scale(self.coord_scale)},
            ],
        }]
        prompt_enc = self.processor.apply_chat_template(
            user_message,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )
        full_enc = self.processor.apply_chat_template(
            user_message + [{"role": "assistant", "content": [{"type": "text", "text": answer}]}],
            add_generation_prompt=False,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )

        prompt_ids = prompt_enc["input_ids"]
        full_ids = full_enc["input_ids"]
        prompt_len = prompt_ids.shape[1]
        if full_ids.shape[1] < prompt_len or not torch.equal(full_ids[:, :prompt_len], prompt_ids):
            raise ValueError(
                "Chat template không giữ prompt làm tiền tố của hội thoại đầy đủ; "
                "không thể mask loss bằng prompt_len một cách an toàn."
            )

        labels = full_ids[0].clone()
        labels[:prompt_len] = -100
        attention_mask = full_enc.get("attention_mask")
        if attention_mask is None:
            attention_mask = torch.ones_like(full_ids)

        example = {
            "input_ids": full_ids[0],
            "attention_mask": attention_mask[0].long(),
            "labels": labels,
            "pixel_values": full_enc.get("pixel_values"),
            "image_grid_thw": full_enc.get("image_grid_thw"),
        }
        mm_token_type_ids = full_enc.get("mm_token_type_ids")
        example["mm_token_type_ids"] = mm_token_type_ids[0] if mm_token_type_ids is not None else None
        return example


def collate_fn(batch):
    def pad_1d(key, padding_value):
        values = [item[key] for item in batch]
        if all(value is None for value in values):
            return None
        if any(value is None for value in values):
            raise ValueError(f"Batch có mm field không đồng nhất: {key}")
        return pad_sequence(values, batch_first=True, padding_value=padding_value)

    out = {
        "input_ids": pad_1d("input_ids", processor.tokenizer.pad_token_id),
        "attention_mask": pad_1d("attention_mask", 0),
        "labels": pad_1d("labels", -100),
    }
    mm_token_type_ids = pad_1d("mm_token_type_ids", 0)
    if mm_token_type_ids is not None:
        out["mm_token_type_ids"] = mm_token_type_ids

    for key in ("pixel_values", "image_grid_thw"):
        values = [item[key] for item in batch]
        if all(value is None for value in values):
            continue
        if any(value is None for value in values):
            raise ValueError(f"Batch có mm field không đồng nhất: {key}")
        out[key] = torch.cat(values, dim=0)
    return out

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
import re
import torch
from peft import LoraConfig, get_peft_model

PROJECTOR_NAME_PARTS = ("visual.merger", "visual.deepstack_merger_list")


def _is_projector_parameter(name: str) -> bool:
    return any(part in name for part in PROJECTOR_NAME_PARTS)


def apply_setup(
    model,
    setup: str,
    lora_r: int = 16,
    lora_alpha: int = 32,
    seed: int = 42,
):
    """Đóng băng/mở đúng các khối theo E1, E2 hoặc E3 và xác nhận target count."""
    setup_upper = setup.upper()
    if setup_upper not in ("E1", "E2", "E3"):
        raise ValueError(f"Setup không hợp lệ: {setup}. Hỗ trợ: 'E1', 'E2', 'E3'.")
    if lora_r < 1 or lora_alpha < 1:
        raise ValueError("lora_r và lora_alpha phải là số nguyên dương")

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    for parameter in model.parameters():
        parameter.requires_grad = False
    model.enable_input_require_grads()

    llm_targets = (
        r"language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|"
        r"mlp\.(gate|up|down)_proj)"
    )
    if setup_upper == "E3":
        target_modules = (
            r".*(language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|"
            r"mlp\.(gate|up|down)_proj)|visual\.blocks\.\d+\.(attn\.(qkv|proj)|"
            r"mlp\.(linear_fc1|linear_fc2)))"
        )
        expected_modules = 348
    else:
        target_modules = r".*" + llm_targets
        expected_modules = 252

    matched_modules = [
        name for name, _ in model.named_modules()
        if re.fullmatch(target_modules, name)
    ]
    if len(matched_modules) != expected_modules:
        raise RuntimeError(
            f"{setup_upper}: regex khớp {len(matched_modules)} module, "
            f"mong đợi {expected_modules}; kiểm tra lại cây module Qwen3-VL."
        )

    peft_model = get_peft_model(
        model,
        LoraConfig(
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=0.05,
            target_modules=target_modules,
            bias="none",
            task_type="CAUSAL_LM",
        ),
    )
    lora_params = sum(
        parameter.numel()
        for name, parameter in peft_model.named_parameters()
        if "lora_" in name and parameter.requires_grad
    )
    if lora_params == 0:
        raise RuntimeError(f"{setup_upper}: PEFT không tạo tham số LoRA trainable.")

    if setup_upper in ("E2", "E3"):
        opened_params = 0
        for name, parameter in peft_model.named_parameters():
            if _is_projector_parameter(name):
                parameter.data = parameter.data.to(torch.float32)
                parameter.requires_grad = True
                opened_params += parameter.numel()
        if opened_params == 0:
            raise RuntimeError(f"{setup_upper}: không tìm thấy projector cần mở.")
        print(f"[{setup_upper}] Projector trainable: {opened_params:,} tham số ở float32.")

    trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in peft_model.parameters())
    print(
        f"[{setup_upper}] Khớp {len(matched_modules)} module LoRA; "
        f"trainable {trainable_params:,}/{total_params:,} ({100 * trainable_params / total_params:.3f}%)."
    )
    return peft_model

## 6. Vòng huấn luyện

In [ ]:
import json
import math
import time
from pathlib import Path

import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm
from transformers import get_cosine_schedule_with_warmup


def train_setup(
    peft_model,
    setup_name: str,
    max_steps: int = 300,
    batch_size: int = 2,
    grad_accum_steps: int = 4,
    coord_scale: str = "norm1000",
    train_fraction: float = 1.0,
    seed: int = 42,
):
    """Huấn luyện SFT theo GUIDE; hỗ trợ subset dữ liệu và thang pixel cho E4."""
    if not torch.cuda.is_available():
        raise RuntimeError("Huấn luyện yêu cầu GPU CUDA; hãy chọn runtime T4 trên Colab.")
    if max_steps < 1 or batch_size < 1 or grad_accum_steps < 1:
        raise ValueError("max_steps, batch_size và grad_accum_steps phải lớn hơn 0")
    if not 0 < train_fraction <= 1:
        raise ValueError("train_fraction phải thuộc (0, 1]")

    setup_name = setup_name.upper()
    train_dataset = RoadDamageDataset(
        "data/splits/train.jsonl", processor, coord_scale=coord_scale
    )
    full_train_size = len(train_dataset)
    if train_fraction < 1.0:
        keep = max(1, math.ceil(len(train_dataset) * train_fraction))
        generator = torch.Generator().manual_seed(seed)
        indices = torch.randperm(len(train_dataset), generator=generator)[:keep].tolist()
        train_dataset = Subset(train_dataset, indices)
        print(f"Dùng {keep}/{full_train_size} ảnh train.")

    loader_generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        generator=loader_generator,
        num_workers=0,
    )
    if len(train_loader) == 0:
        raise ValueError("Tập train rỗng")

    trainable_parameters = [p for p in peft_model.parameters() if p.requires_grad]
    trainable_count = sum(p.numel() for p in trainable_parameters)
    if trainable_count == 0:
        raise RuntimeError("Không có tham số trainable; hãy gọi apply_setup trước khi train.")

    optimizer = AdamW(trainable_parameters, lr=2e-4, weight_decay=0.0)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, int(max_steps * 0.05)),
        num_training_steps=max_steps,
    )
    scaler = torch.amp.GradScaler("cuda")

    peft_model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
    peft_model.config.use_cache = False
    peft_model.train()
    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    step = 0
    loader_iter = iter(train_loader)
    progress = tqdm(total=max_steps, desc=f"Huấn luyện {setup_name}")
    while step < max_steps:
        optimizer.zero_grad(set_to_none=True)
        step_loss = 0.0

        for _ in range(grad_accum_steps):
            try:
                batch = next(loader_iter)
            except StopIteration:
                loader_iter = iter(train_loader)
                batch = next(loader_iter)

            batch = {
                key: value.to("cuda") if isinstance(value, torch.Tensor) else value
                for key, value in batch.items()
            }
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                loss = peft_model(**batch).loss / grad_accum_steps
            if not torch.isfinite(loss):
                raise RuntimeError(f"Loss không hữu hạn (NaN/Inf) tại step {step}; dừng train.")

            scaler.scale(loss).backward()
            step_loss += loss.item() * grad_accum_steps

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_parameters, 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        step += 1
        progress.update(1)
        progress.set_postfix({"loss": f"{step_loss:.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})

        if step % 100 == 0:
            checkpoint_path = Path(f"work/{setup_name}_adapter_step{step}")
            checkpoint_path.mkdir(parents=True, exist_ok=True)
            peft_model.save_pretrained(checkpoint_path)
            checkpoint_projector = {
                name: parameter.detach().cpu().clone()
                for name, parameter in peft_model.named_parameters()
                if _is_projector_parameter(name) and parameter.requires_grad
            }
            if checkpoint_projector:
                torch.save(checkpoint_projector, checkpoint_path / "projector_weights.pt")
            if "backup_to_drive" in globals():
                backup_to_drive(str(checkpoint_path), f"{setup_name}_adapter_step{step}")

    progress.close()
    torch.cuda.synchronize()
    train_minutes = (time.time() - start_time) / 60.0
    peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

    save_path = Path(f"work/{setup_name}_adapter")
    save_path.mkdir(parents=True, exist_ok=True)
    peft_model.save_pretrained(save_path)
    projector_weights = {
        name: parameter.detach().cpu().clone()
        for name, parameter in peft_model.named_parameters()
        if _is_projector_parameter(name) and parameter.requires_grad
    }
    if projector_weights:
        torch.save(projector_weights, save_path / "projector_weights.pt")

    meta_info = {
        "trainable_params": int(trainable_count),
        "peak_vram_gb": round(float(peak_vram_gb), 2),
        "train_minutes": round(float(train_minutes), 2),
        "coord_scale": coord_scale,
        "train_fraction": float(train_fraction),
        "max_steps": int(max_steps),
        "projector_saved": bool(projector_weights),
    }
    run_file = Path(f"work/run_{setup_name.lower()}.json")
    run_file.parent.mkdir(parents=True, exist_ok=True)
    run_file.write_text(json.dumps(meta_info, indent=2), encoding="utf-8")
    print(f"Đã lưu adapter tại {save_path}; metadata: {meta_info}")

    if "backup_to_drive" in globals():
        backup_to_drive(str(save_path), f"{setup_name}_adapter")
        backup_to_drive(str(run_file), f"{setup_name}_run.json")
    return meta_info


def train_experiment(
    setup_name: str,
    max_steps: int = 300,
    batch_size: int = 2,
    grad_accum_steps: int = 4,
):
    """Nạp model gốc mới, áp dụng E1/E2/E3, train rồi giải phóng VRAM."""
    setup_name = setup_name.upper()
    if setup_name not in ("E1", "E2", "E3"):
        raise ValueError("setup_name phải là E1, E2 hoặc E3")
    base_model = load_base_model()
    peft_model = apply_setup(base_model, setup_name, seed=42)
    meta = train_setup(
        peft_model,
        setup_name,
        max_steps=max_steps,
        batch_size=batch_size,
        grad_accum_steps=grad_accum_steps,
        seed=42,
    )
    del peft_model, base_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return meta

# Ví dụ: train_experiment("E1"), sau đó run_eval("E1", meta, split="val") để gỡ lỗi.
# Chỉ gọi run_eval(..., split="test") khi sẵn sàng chấm test.

## 7. Suy luận và chấm E1–E3

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import torch
from PIL import Image
from peft import PeftModel


def _load_projector_weights(model, projector_path: Path, required: bool = False) -> bool:
    if not projector_path.exists():
        if required:
            raise FileNotFoundError(
                f"Thiếu projector_weights.pt tại {projector_path}; kết quả sẽ không khớp setup đã train."
            )
        return False

    # Phục hồi dtype float32 đã dùng khi train projector E2/E3. Inference chạy trong autocast fp16.
    for name, parameter in model.named_parameters():
        if _is_projector_parameter(name):
            parameter.data = parameter.data.to(torch.float32)
    weights = torch.load(projector_path, map_location="cpu")
    if not isinstance(weights, dict) or not weights:
        raise ValueError(f"Checkpoint projector rỗng hoặc sai định dạng: {projector_path}")
    if not all(_is_projector_parameter(name) for name in weights):
        raise ValueError("Checkpoint projector chứa key ngoài visual.merger/deepstack_merger_list.")

    expected_keys = {
        name for name, _ in model.named_parameters() if _is_projector_parameter(name)
    }
    if set(weights) != expected_keys:
        missing = sorted(expected_keys - set(weights))
        extra = sorted(set(weights) - expected_keys)
        raise RuntimeError(
            f"Projector checkpoint không đầy đủ: thiếu={missing[:5]}, ngoài dự kiến={extra[:5]}"
        )
    result = model.load_state_dict(weights, strict=False)
    if result.unexpected_keys:
        raise RuntimeError(f"Projector keys không được nạp: {result.unexpected_keys[:5]}")
    print(f"Đã nạp và xác nhận {len(weights)} tensors projector từ {projector_path}.")
    return True


def run_eval(
    setup_name: str,
    meta_info: dict | None = None,
    split: str = "test",
    batch_size: int = 1,
    coord_scale: str = "norm1000",
):
    """Nạp adapter/projector, sinh dự đoán theo batch và gọi trọng tài scripts/score.py."""
    setup_name = setup_name.upper()
    if split not in ("test", "val"):
        raise ValueError("split chỉ nhận 'test' hoặc 'val'")
    if coord_scale not in ("norm1000", "pixel"):
        raise ValueError(f"coord_scale không hợp lệ: {coord_scale}")

    adapter_path = Path(f"work/{setup_name}_adapter")
    if not adapter_path.exists():
        raise FileNotFoundError(f"Chưa tìm thấy adapter tại {adapter_path}; hãy train trước.")
    if meta_info is None:
        run_file = Path(f"work/run_{setup_name.lower()}.json")
        if run_file.exists():
            meta_info = json.loads(run_file.read_text(encoding="utf-8"))

    print(f"--- ĐÁNH GIÁ {setup_name} trên split={split}, coord={coord_scale} ---")
    base_model = load_base_model()
    eval_model = PeftModel.from_pretrained(base_model, str(adapter_path))
    projector_path = adapter_path / "projector_weights.pt"
    projector_required = setup_name in ("E2", "E3") or bool(
        meta_info and meta_info.get("projector_saved")
    )
    _load_projector_weights(eval_model, projector_path, required=projector_required)
    eval_model.eval()

    split_file = Path(f"data/splits/{split}.jsonl")
    with split_file.open("r", encoding="utf-8") as f:
        entries = [json.loads(line) for line in f]
    prediction_path = Path(f"work/{setup_name.lower()}.jsonl")
    write_predictions_jsonl(
        eval_model,
        entries,
        prediction_path,
        system_prompt_for_scale(coord_scale),
        batch_size=batch_size,
        desc=f"Suy luận {setup_name} ({split})",
    )

    del eval_model, base_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    evidence_out = Path(f"report/evidence/{setup_name.lower()}.json")
    evidence_out.parent.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable,
        "scripts/score.py",
        "--pred", str(prediction_path),
        "--name", setup_name,
        "--split", split,
        "--coord", coord_scale,
        "--out", str(evidence_out),
    ]
    if meta_info:
        for key in ("trainable_params", "peak_vram_gb", "train_minutes"):
            if key in meta_info:
                command.extend(["--meta", f"{key}={meta_info[key]}"])

    subprocess.run(command, check=True)
    print(f"Đã chấm điểm xong {setup_name}; bằng chứng tại {evidence_out}")
    return evidence_out

## 8. E4: Thí nghiệm Ablation (Cô lập một biến số)


In [ ]:
# E4: mỗi biến thể được train lại từ model gốc với cùng recipe; chỉ thay biến đang khảo sát.
# Mặc định chọn E1 làm setup đối chứng. Đổi base_setup_name thành E2/E3 nếu đó là setup tốt nhất.


def _run_e4_variant(
    base_setup_name: str,
    run_name: str,
    max_steps: int = 300,
    lora_rank: int = 16,
    coord_scale: str = "norm1000",
    train_fraction: float = 1.0,
):
    base_setup_name = base_setup_name.upper()
    if base_setup_name not in ("E1", "E2", "E3"):
        raise ValueError("base_setup_name phải là E1, E2 hoặc E3")

    base_model = load_base_model()
    peft_model = apply_setup(
        base_model,
        base_setup_name,
        lora_r=lora_rank,
        lora_alpha=2 * lora_rank,
    )
    meta = train_setup(
        peft_model,
        run_name,
        max_steps=max_steps,
        coord_scale=coord_scale,
        train_fraction=train_fraction,
    )
    del peft_model, base_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return run_eval(
        run_name,
        meta_info=meta,
        split="test",
        coord_scale=coord_scale,
    )


def run_e4_ablation_pixel(base_setup_name: str = "E1", max_steps: int = 300):
    """Train E4 pixel: pixel trong nhãn, prompt và trọng tài; so với base_setup norm1000."""
    return _run_e4_variant(
        base_setup_name=base_setup_name,
        run_name="E4_coord_scale_pixel",
        max_steps=max_steps,
        lora_rank=16,
        coord_scale="pixel",
    )


def run_e4_ablation_rank(rank: int, base_setup_name: str = "E1", max_steps: int = 300):
    """Train một giá trị rank trong cấu hình E4; alpha luôn bằng 2*r theo configs/e4.yaml."""
    if rank not in (4, 64):
        raise ValueError("configs/e4.yaml yêu cầu rank=4 hoặc rank=64")
    return _run_e4_variant(
        base_setup_name=base_setup_name,
        run_name=f"E4_rank_{rank}",
        max_steps=max_steps,
        lora_rank=rank,
        coord_scale="norm1000",
    )


def run_e4_rank_pair(base_setup_name: str = "E1", max_steps: int = 300):
    """Chạy đủ r=4 và r=64; so sánh với evidence của base_setup_name (r=16)."""
    results = {}
    for rank in (4, 64):
        results[rank] = run_e4_ablation_rank(
            rank=rank,
            base_setup_name=base_setup_name,
            max_steps=max_steps,
        )
    return results


# Chọn đúng một biến cho E4 và chạy các giá trị tương ứng:
# run_e4_ablation_pixel(base_setup_name="E1", max_steps=300)
# run_e4_rank_pair(base_setup_name="E1", max_steps=300)
# run_e4_data_size_pair(base_setup_name="E1", max_steps=300)


def run_e4_ablation_data_size(
    train_fraction: float,
    base_setup_name: str = "E1",
    max_steps: int = 300,
):
    """Train một mức data_size trong configs/e4.yaml; chỉ đổi lượng ảnh train."""
    if train_fraction not in (0.1, 1.0):
        raise ValueError("configs/e4.yaml yêu cầu train_fraction=0.1 hoặc 1.0")
    percent = int(round(train_fraction * 100))
    return _run_e4_variant(
        base_setup_name=base_setup_name,
        run_name=f"E4_data_size_{percent}",
        max_steps=max_steps,
        lora_rank=16,
        coord_scale="norm1000",
        train_fraction=train_fraction,
    )


def run_e4_data_size_pair(base_setup_name: str = "E1", max_steps: int = 300):
    """Chạy 10% và 100% data_size với cùng setup/hyperparameters."""
    return {
        fraction: run_e4_ablation_data_size(
            train_fraction=fraction,
            base_setup_name=base_setup_name,
            max_steps=max_steps,
        )
        for fraction in (0.1, 1.0)
    }

## 9. E5 (Tuỳ chọn, +10 điểm): GRPO với Reward IoU

> E5 nạp adapter tốt nhất từ E1–E3, giữ projector đã học nếu có, rồi cập nhật LoRA bằng GRPO. Trong [tài liệu TRL 1.9.2 về VLM training](https://github.com/huggingface/trl/blob/v1.9.2/docs/source/grpo_trainer.md#vision-language-model-vlm-training), Qwen3-VL chưa nằm trong danh sách VLM được nêu là đã kiểm thử; hãy kiểm tra một bước train trên Colab trước khi chạy đủ 60 bước.

In [ ]:
# E5: reward IoU thuần Python, kiểm tra trước khi khởi tạo GRPOTrainer.
import json
import random
from pathlib import Path

import torch
from PIL import Image
from scripts.score import iou_xyxy, parse_output


def compute_iou_reward(completion: str, gt_boxes: list[dict]) -> float:
    """Reward theo GUIDE: JSON hỏng=0; rỗng+rỗng=1; ghép cùng lớp với IoU>=0.5."""
    pred_boxes = parse_output(completion)
    if pred_boxes is None:
        return 0.0
    if not gt_boxes and not pred_boxes:
        return 1.0

    pairs = []
    for gt_index, gt in enumerate(gt_boxes):
        gt_bbox = gt.get("bbox", gt.get("bbox_2d"))
        for pred_index, pred in enumerate(pred_boxes):
            if gt.get("label") != pred.get("label"):
                continue
            pred_bbox = pred.get("bbox", pred.get("bbox_2d"))
            overlap = iou_xyxy(gt_bbox, pred_bbox)
            if overlap >= 0.5:
                pairs.append((overlap, gt_index, pred_index))

    # Tie-break ổn định để cùng input luôn cho cùng matching.
    pairs.sort(key=lambda pair: (-pair[0], pair[1], pair[2]))
    used_gt, used_pred = set(), set()
    total_iou = 0.0
    for overlap, gt_index, pred_index in pairs:
        if gt_index not in used_gt and pred_index not in used_pred:
            used_gt.add(gt_index)
            used_pred.add(pred_index)
            total_iou += overlap

    denominator = max(len(gt_boxes), len(pred_boxes), 1)
    return round(float(0.2 + 0.8 * total_iou / denominator), 4)


def _completion_to_text(completion) -> str:
    """Lấy nội dung assistant từ dạng chuỗi hoặc conversational completion của TRL."""
    if isinstance(completion, str):
        return completion
    if isinstance(completion, dict):
        messages = [completion]
    elif isinstance(completion, (list, tuple)):
        messages = list(completion)
    else:
        return ""

    assistant_messages = [
        message for message in messages
        if isinstance(message, dict) and message.get("role") == "assistant"
    ]
    message = assistant_messages[-1] if assistant_messages else (messages[-1] if messages else {})
    content = message.get("content", "") if isinstance(message, dict) else ""
    if isinstance(content, str):
        return content
    if isinstance(content, (list, tuple)):
        return "".join(
            block.get("text", "")
            for block in content
            if isinstance(block, dict) and isinstance(block.get("text", ""), str)
        )
    return ""


def test_compute_iou_reward():
    gt = [{"label": "pothole", "bbox": [100, 100, 200, 200]}]
    perfect = '[{"label":"pothole","bbox_2d":[100,100,200,200]}]'
    malformed = "Here is pothole at [100, 100]"
    wrong_class = '[{"label":"crack","bbox_2d":[100,100,200,200]}]'
    extra_box = (
        '[{"label":"pothole","bbox_2d":[100,100,200,200]},'
        '{"label":"pothole","bbox_2d":[500,500,600,600]}]'
    )

    assert compute_iou_reward(perfect, gt) == 1.0
    assert compute_iou_reward(malformed, gt) == 0.0
    assert compute_iou_reward(wrong_class, gt) == 0.2
    assert compute_iou_reward("[]", []) == 1.0
    assert 0.2 < compute_iou_reward(extra_box, gt) < 1.0
    assert _completion_to_text([{"role": "assistant", "content": perfect}]) == perfect
    print("Tất cả kiểm tra reward IoU đã qua.")


test_compute_iou_reward()


def train_and_eval_e5_grpo(best_adapter_name: str = "E1", max_steps: int = 60):
    """Train GRPO 60 bước từ adapter tốt nhất, lưu adapter/projector rồi chấm test."""
    if not torch.cuda.is_available():
        raise RuntimeError("E5 yêu cầu GPU CUDA; hãy chạy trong Colab T4.")
    if max_steps < 1:
        raise ValueError("max_steps phải lớn hơn 0")

    from datasets import Dataset
    from peft import PeftModel
    from trl import GRPOConfig, GRPOTrainer
    import time

    best_adapter_name = best_adapter_name.upper()
    if best_adapter_name not in ("E1", "E2", "E3"):
        raise ValueError("best_adapter_name phải là E1, E2 hoặc E3")
    adapter_src = Path(f"work/{best_adapter_name}_adapter")
    if not adapter_src.exists():
        raise FileNotFoundError(f"Không tìm thấy adapter khởi tạo: {adapter_src}")

    base_model = load_base_model()
    train_model = PeftModel.from_pretrained(
        base_model,
        str(adapter_src),
        is_trainable=True,
    )
    source_projector = adapter_src / "projector_weights.pt"
    projector_was_saved = _load_projector_weights(
        train_model,
        source_projector,
        required=best_adapter_name in ("E2", "E3"),
    )

    with open("data/splits/train.jsonl", "r", encoding="utf-8") as f:
        all_train_rows = [json.loads(line) for line in f]
    rng = random.Random(42)
    train_rows = rng.sample(all_train_rows, k=min(200, len(all_train_rows)))
    grpo_rows = []
    for item in train_rows:
        with Image.open(Path("data") / item["image_path"]) as source_image:
            image = source_image.convert("RGB")
        grpo_rows.append({
            "prompt": [{"role": "user", "content": SYSTEM_PROMPT}],
            "image": image,
            "gt_boxes": [yolo_to_box(obj["label"], obj["yolo"]) for obj in item["objects"]],
        })
    if not grpo_rows:
        raise ValueError("Không có ảnh train để tạo tập GRPO")
    grpo_dataset = Dataset.from_list(grpo_rows)

    def reward_function(completions, gt_boxes, **kwargs):
        if len(completions) != len(gt_boxes):
            raise ValueError(
                f"TRL truyền {len(completions)} completions nhưng {len(gt_boxes)} ground truths"
            )
        return [
            compute_iou_reward(_completion_to_text(completion), gt)
            for completion, gt in zip(completions, gt_boxes, strict=True)
        ]

    output_dir = Path("work/E5_adapter")
    grpo_config = GRPOConfig(
        output_dir=str(output_dir),
        learning_rate=5.0e-6,
        num_generations=4,
        max_completion_length=128,
        max_steps=max_steps,
        beta=0.0,
        per_device_train_batch_size=1,
        # Effective batch=4 để chia hết cho num_generations=4 theo TRL 1.9.2.
        gradient_accumulation_steps=4,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=5,
        save_strategy="no",
        remove_unused_columns=False,
        use_vllm=False,
        report_to="none",
        dataloader_num_workers=0,
    )
    trainer = GRPOTrainer(
        model=train_model,
        reward_funcs=reward_function,
        args=grpo_config,
        train_dataset=grpo_dataset,
        processing_class=processor,
    )

    trainable_count = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
    if trainable_count == 0:
        raise RuntimeError("GRPOTrainer không nhận được tham số trainable; dừng để tránh chạy rỗng.")
    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()
    trainer.train()
    torch.cuda.synchronize()
    train_minutes = (time.time() - start_time) / 60.0
    peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

    output_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(output_dir))
    projector_weights = {
        name: parameter.detach().cpu().clone()
        for name, parameter in trainer.model.named_parameters()
        if _is_projector_parameter(name)
    }
    if projector_weights:
        torch.save(projector_weights, output_dir / "projector_weights.pt")

    meta = {
        "trainable_params": int(trainable_count),
        "peak_vram_gb": round(float(peak_vram_gb), 2),
        "train_minutes": round(float(train_minutes), 2),
        "projector_saved": bool(projector_weights),
        "projector_loaded_from": best_adapter_name if projector_was_saved else None,
    }
    run_file = Path("work/run_e5.json")
    run_file.write_text(json.dumps(meta, indent=2), encoding="utf-8")

    del trainer, train_model, base_model
    torch.cuda.empty_cache()
    if "backup_to_drive" in globals():
        backup_to_drive(str(output_dir), "E5_adapter")
        backup_to_drive(str(run_file), "E5_run.json")

    return run_eval("E5", meta_info=meta, split="test", batch_size=1)


print("Reward IoU đã sẵn sàng; train_and_eval_e5_grpo() sẽ train và chấm E5 khi được gọi.")